In [ ]:
%config InlineBackend.figure_formats = ['svg']
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import sparse
import scipy.sparse.linalg as spla
from functools import reduce

In [ ]:
plt.style.use("dark_background")
# mpl.rcParams["figure.facecolor"] = 'none'
mpl.rcParams["axes.facecolor"] = "none"
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["figure.figsize"] = [12.8, 9.6]

# Notes

Spec for mixed BC:

$$\begin{align}
    -\triangle u &= f,            \qquad x \in \Omega \\
    \alpha u + \partial_n u &= g, \qquad x \in \partial \Omega
\end{align}$$

In [ ]:
from collections import namedtuple

Dimension = namedtuple(
    "Dimension", ["low", "high", "num_divs", "low_robin", "high_robin"]
)
BoxDomain = namedtuple("BoxDomain", ["dimensions"])

In [ ]:
def fdm_neg_lapl_1d_mat(dim: Dimension):
    interior_size = dim.num_divs - 1
    full_size = (
        interior_size + int(dim.low_robin is not None) + int(dim.high_robin is not None)
    )
    dim_len = dim.high - dim.low
    h = dim_len / dim.num_divs
    invhsqr = 1 / (h * h)

    return (
        sparse.block_array(
            (
                [
                    [
                        sparse.csr_array(
                            ([2.0 * (1 + h * dim.low_robin), -2.0], [0, 1], [0, 2]),
                            shape=(1, full_size),
                        )
                    ]
                ]
                if dim.low_robin is not None
                else [[None]]
            )
            + (
                [
                    [
                        sparse.diags_array(
                            [2.0, -1, -1],
                            offsets=np.array([0, 1, -1])
                            + int(dim.low_robin is not None),
                            shape=(interior_size, full_size),
                        )
                    ]
                ]
            )
            + (
                [
                    [
                        sparse.csr_array(
                            (
                                [2.0 * (1 + h * dim.high_robin), -2.0],
                                [full_size - 1, full_size - 2],
                                [0, 2],
                            ),
                            shape=(1, full_size),
                        )
                    ]
                ]
                if dim.high_robin is not None
                else [[None]]
            )
        )
        * invhsqr
    )


def fdm_neg_lapl_unif_disc_mat(dom: BoxDomain):
    return reduce(
        sparse.kronsum,
        (fdm_neg_lapl_1d_mat(dim) for dim in reversed(dom.dimensions)),
    )


domain = BoxDomain(
    [
        Dimension(0.0, 1.0, 2, 0.0, 0.0),
        Dimension(0.0, 1.0, 2, 0.0, 0.0),
        Dimension(0.0, 1.0, 2, 0.0, 0.0),
    ]
)

fig, ax = plt.subplots()
ax: plt.Axes = ax

ax_im = ax.matshow(
    fdm_neg_lapl_unif_disc_mat(domain).toarray(),
    norm=mpl.colors.CenteredNorm(),
    cmap="berlin",
)
fig.colorbar(ax_im)
plt.show()